# AUTONOMOUS MOVIE STUDIO\n## Real Qwen GPU Validation\nRun all cells in a fresh GPU-enabled Colab runtime. The notebook fails on fallback, missing artifacts, or an invalid render.

In [ ]:
import os, subprocess, sys, json, pathlib, re, time\nREPO = 'https://github.com/asdfhgds/automovies.git'\nBRANCH = 'asdfhgds-autonomous-movie-studio-spec'\nMODEL = 'Qwen/Qwen2.5-1.5B-Instruct'  # configurable; suitable for a T4\ngpu = subprocess.run(['nvidia-smi'], text=True, capture_output=True)\nprint(gpu.stdout)\nif gpu.returncode: raise RuntimeError('GPU REQUIRED — Runtime → Change runtime type → GPU')

In [ ]:
subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,'automovies'], check=True)\nos.chdir('automovies')\nsubprocess.run(['bash','scripts/colab_setup.sh'], check=True)\nos.environ.update({'STUDIO_PROFILE':'colab-gpu','REQUIRE_REAL_LLM':'true','DIRECTOR_PROVIDER':'qwen','DIRECTOR_MODEL':MODEL,'DIRECTOR_DEVICE':'cuda','SCRIPT_PROVIDER':'qwen','SCRIPT_MODEL':MODEL,'SCRIPT_DEVICE':'cuda','CREATIVE_DIRECTOR_ENABLED':'true'})\nsubprocess.run([sys.executable,'src/main.py','doctor'], check=True)

In [ ]:
# Create a legal local fixture and discover its project ID without manual copying.\nsubprocess.run([sys.executable,'tests/fixtures/generate_test_fixture.py','tests/fixtures/test_speech.mp4','Short legal GPU test'], check=True)\ncreated = subprocess.check_output([sys.executable,'src/main.py','init','--title','Qwen GPU Validation','--source','tests/fixtures/test_speech.mp4'], text=True)\nprint(created)\nmatch = re.search(r'Initialized project ([0-9a-f-]{36})', created)\nif not match: raise RuntimeError('Could not determine project ID')\nPROJECT_ID = match.group(1)\nstarted = time.time()\nsubprocess.run([sys.executable,'src/main.py','run','--project-id',PROJECT_ID], check=True)\npipeline_seconds = round(time.time() - started, 2)\nproject = pathlib.Path('data') / PROJECT_ID\nprint('Pipeline seconds:', pipeline_seconds)

In [ ]:
# Strict artifact and provider validation: no fallback may pass this cell.\nrequired = ['transcripts/transcript.json','scenes/scene_index.json','scenes/scene_ranking.json','scenes/selected_scenes.json','director_plan.json','script.json','timeline/timeline.json','renders/final_render.mp4','reports/qc_report.json']\nmissing = [item for item in required if not (project/item).exists()]\nfor item in required: print(f'{item:38} {"PASS" if item not in missing else "FAIL"}')\nif missing: raise AssertionError(f'Missing artifacts: {missing}')\nplan = json.loads((project/'director_plan.json').read_text())\nscript = json.loads((project/'script.json').read_text())\ndirector_meta, script_meta = plan.get('provider_metadata', {}), script.get('provider_metadata', {})\nassert director_meta.get('model') == MODEL and director_meta.get('device') == 'cuda', director_meta\nassert script.get('provider') == 'qwen' and script_meta.get('model') == MODEL and script_meta.get('device') == 'cuda', script_meta\nassert len(plan.get('all_concepts', [])) >= 3, 'Qwen did not produce at least three concepts'\nscene_ids = {scene['scene_id'] for scene in json.loads((project/'scenes/scene_index.json').read_text())}\nassert all(item['scene_id'] in scene_ids for item in json.loads((project/'scenes/selected_scenes.json').read_text()))\nprobe = subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration:stream=codec_name,width,height,r_frame_rate,sample_rate','-of','json',str(project/'renders/final_render.mp4')], text=True)\nmedia = json.loads(probe); print(json.dumps(media, indent=2))\ncodecs = {stream.get('codec_name') for stream in media.get('streams', [])}\nassert {'h264','aac'} <= codecs and float(media['format']['duration']) > 0, codecs

In [ ]:
report = pathlib.Path('GPU_VALIDATION_FINAL_REPORT.md')\nreport.write_text(f'''# GPU Validation\n\nProject: {PROJECT_ID}\n\nModel: {MODEL}\nDevice: cuda\nPipeline seconds: {pipeline_seconds}\n\nDirector: PASS\nScript: PASS\nScene selection: PASS\nTimeline/render/QC: PASS\n\nReal Qwen execution: PASS\n''')\nprint(report.read_text())\nfrom google.colab import files\nfiles.download(str(project/'renders/final_render.mp4'))\nfiles.download(str(report))